In [20]:
import pandas as pd

In [21]:
# 1. Загрузка данных
# Используем кодировку 'ISO-8859-1', так как в этом датасете часто встречаются спецсимволы
df = pd.read_csv('OnlineRetail.csv', encoding='ISO-8859-1')

In [22]:
print(f"Размер исходного датасета: {df.shape[0]} строк, {df.shape[1]} колонок.")
print("-" * 50)

Размер исходного датасета: 541909 строк, 8 колонок.
--------------------------------------------------


In [23]:
# ==========================================
# ШАГ 1: Валидация типов данных
# ==========================================
print("\n[1] ПРОВЕРКА ТИПОВ ДАННЫХ:")
print(df.info())

# Переводим InvoiceDate в формат даты для удобства дальнейшей работы
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"\nДиапазон дат в датасете: с {df['InvoiceDate'].min()} по {df['InvoiceDate'].max()}")
print("-" * 50)


[1] ПРОВЕРКА ТИПОВ ДАННЫХ:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB
None

Диапазон дат в датасете: с 2010-12-01 08:26:00 по 2011-12-09 12:50:00
--------------------------------------------------


In [24]:

# ==========================================
# ШАГ 2: Проверка пропусков в CustomerID
# ==========================================
null_customers = df['CustomerID'].isnull().sum()
total_rows = len(df)
pct_null_customers = (null_customers / total_rows) * 100

print(f"\n[2] ПРОВЕРКА ПРОПУЩЕННЫХ КЛИЕНТОВ (CustomerID):")
print(f"Количество пропусков (гостевые заказы): {null_customers}")
print(f"Доля гостевых заказов от общего числа: {pct_null_customers:.2f}%")
print("-" * 50)

# ==========================================
# ШАГ 3: Проверка отрицательных значений
# ==========================================
# 3.1. Анализ отрицательных количеств (Quantity)
neg_qty_df = df[df['Quantity'] < 0]
total_neg_qty = len(neg_qty_df)
pct_neg_qty = (total_neg_qty / total_rows) * 100

print(f"\n[3] АНАЛИЗ ОТРИЦАТЕЛЬНЫХ КОЛИЧЕСТВ (Quantity):")
print(f"Всего строк с отрицательным Quantity: {total_neg_qty} ({pct_neg_qty:.2f}%)")

# Разделяем на отмены (начинаются на 'C') и прочие корректировки
df['InvoiceNo_Str'] = df['InvoiceNo'].astype(str)
cancelled_orders = df[df['InvoiceNo_Str'].str.startswith('C', na=False)]
neg_no_c = df[(df['Quantity'] < 0) & (~df['InvoiceNo_Str'].str.startswith('C', na=False))]

print(f" └─ Из них официальных отмен/возвратов (код 'C...'): {len(cancelled_orders)}")
print(f" └─ Административных списаний/корректировок (без кода 'C'): {len(neg_no_c)}")

# 3.2. Проверка цен (UnitPrice)
neg_price_df = df[df['UnitPrice'] < 0]
zero_price_df = df[df['UnitPrice'] == 0]

print(f"\nАНАЛИЗ ЦЕН (UnitPrice):")
print(f"Строк с отрицательной ценой (например, корректировка долга): {len(neg_price_df)}")
if len(neg_price_df) > 0:
    print(neg_price_df[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice']])
print(f"Строк с нулевой ценой (подарки, промо или списания): {len(zero_price_df)} ({len(zero_price_df)/total_rows*100:.2f}%)")
print("-" * 50)

# ==========================================
# РЕЗЮМЕ ДЛЯ СОХРАНЕНИЯ ДАННЫХ
# ==========================================
print("\n[ИТОГОВОЕ РЕШЕНИЕ]: Данные сохранены в исходном виде.")
print(f"Текущее количество строк для анализа: {len(df)}")


[2] ПРОВЕРКА ПРОПУЩЕННЫХ КЛИЕНТОВ (CustomerID):
Количество пропусков (гостевые заказы): 135080
Доля гостевых заказов от общего числа: 24.93%
--------------------------------------------------

[3] АНАЛИЗ ОТРИЦАТЕЛЬНЫХ КОЛИЧЕСТВ (Quantity):
Всего строк с отрицательным Quantity: 10624 (1.96%)
 └─ Из них официальных отмен/возвратов (код 'C...'): 9288
 └─ Административных списаний/корректировок (без кода 'C'): 1336

АНАЛИЗ ЦЕН (UnitPrice):
Строк с отрицательной ценой (например, корректировка долга): 2
       InvoiceNo      Description  Quantity  UnitPrice
299983   A563186  Adjust bad debt         1  -11062.06
299984   A563187  Adjust bad debt         1  -11062.06
Строк с нулевой ценой (подарки, промо или списания): 2515 (0.46%)
--------------------------------------------------

[ИТОГОВОЕ РЕШЕНИЕ]: Данные сохранены в исходном виде.
Текущее количество строк для анализа: 541909


In [25]:
import pandas as pd
import matplotlib.pyplot as plt

# отображать все столбцы
pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!
